### Attribute Write Accessors — Advanced Guided Problems

This notebook continues the **same topic** as the provided notebook, but with new, more advanced problems.

The style is deliberately tutorial-like:

- we introduce one question at a time;
- we run a small experiment;
- we inspect what happened;
- we explain why;
- then we extend the idea.

The supplied notebook's prose refers to `__setattribute__`, while its working examples define `__setattr__`. In the exercises below we follow the behavior demonstrated by the source code and use `__setattr__`.

We will stay focused on the ideas developed in the source notebook:

- intercepting instance attribute writes;
- delegating writes safely;
- avoiding recursion;
- understanding what happens when data descriptors are involved;
- distinguishing instance writes from class writes;
- using a metaclass when the **class object itself** needs write interception.

We will make the problems more demanding by combining those ideas with inheritance, initialization state, caching, observers, slots, and class-level policies.

Let's start with a very small trace.

Before building anything complicated, it is useful to see exactly when our write hook runs.

## Problem 1 — Count writes without causing recursion

Suppose we want every instance to count how many public attribute assignments have happened.

At first this sounds easy: increment a counter inside `__setattr__`.

But there is an immediate problem.

If we write:

```python
self.write_count += 1
```

inside `__setattr__`, that assignment itself calls `__setattr__` again.

So the first problem is:

> How can we maintain internal bookkeeping without recursively calling our own write hook?

We will first build the object so that the internal counter exists **before** normal assignments begin.

For that bootstrap step, we bypass our custom hook and call the base object implementation directly.

In [1]:
class WriteCounter:
    def __init__(self):
        object.__setattr__(self, "_write_count", 0)

    @property
    def write_count(self):
        return self._write_count

    def __setattr__(self, name, value):
        if not name.startswith("_"):
            current = object.__getattribute__(self, "_write_count")
            object.__setattr__(self, "_write_count", current + 1)

        super().__setattr__(name, value)

Now let's perform a few writes.

Notice that we are not doing anything special when assigning `name`, `score`, or `active`.

Those are ordinary instance writes, so they all pass through `__setattr__`.

In [2]:
counter = WriteCounter()

counter.name = "Guido"
counter.score = 100
counter.active = True

counter.write_count

3

The counter should now be `3`.

The important point is that `_write_count` was updated with `object.__setattr__`, so updating the counter did **not** re-enter our custom `__setattr__`.

Let's verify the complete state.

In [3]:
assert counter.write_count == 3
assert counter.__dict__ == {
    "_write_count": 3,
    "name": "Guido",
    "score": 100,
    "active": True,
}

counter.__dict__

{'_write_count': 3, 'name': 'Guido', 'score': 100, 'active': True}

This gives us a useful pattern:

> Public assignment goes through the custom policy. Internal bookkeeping can use a deliberate bypass.

We will use this pattern several times later in the notebook.

## Problem 2 — Observe the order between `__setattr__` and a property setter

The source notebook shows that a **data descriptor** participates in assignment.

A property with a setter is also a data descriptor.

So let's ask a more precise question:

> If a class has both `__setattr__` and a property setter, which one runs first?

We will record events instead of just printing them.

That makes the order easy to inspect and easy to test.

In [4]:
class Temperature:
    def __init__(self):
        object.__setattr__(self, "events", [])
        self.celsius = 20

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        self.events.append(("property setter", value))
        object.__setattr__(self, "_celsius", float(value))

    def __setattr__(self, name, value):
        if name != "events":
            self.events.append(("__setattr__", name, value))
        super().__setattr__(name, value)

Now instantiate the class.

The assignment `self.celsius = 20` first reaches our custom `__setattr__`.

Then `super().__setattr__` sees that `celsius` is managed by a data descriptor — the property — and calls the property's setter.

In [5]:
t = Temperature()

t.events

[('__setattr__', 'celsius', 20), ('property setter', 20)]

We expect two events.

The outer write hook sees the assignment first.

Then the property setter handles the managed field.

In [6]:
assert t.events == [
    ("__setattr__", "celsius", 20),
    ("property setter", 20),
]

assert t.celsius == 20.0

t.__dict__

{'events': [('__setattr__', 'celsius', 20), ('property setter', 20)],
 '_celsius': 20.0}

So a useful mental model is:

1. `obj.name = value` enters the class's `__setattr__`.
2. If that method delegates with `super().__setattr__`, the normal attribute machinery continues.
3. If `name` is controlled by a data descriptor, that descriptor's `__set__` method is invoked.

A property setter is one concrete example of that descriptor behavior.

## Problem 3 — Build a descriptor and trace the same path explicitly

Now let's replace the property with our own data descriptor.

We will make a descriptor that stores a percentage in the range `0` to `100`.

The host class will still have its own `__setattr__`.

This lets us see the same two-layer write path without relying on `property`.

First, the descriptor.

It will:

- remember its attribute name;
- validate values;
- store the actual number under a separate key in the instance dictionary.

In [7]:
class Percentage:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        instance.trace.append(("descriptor __set__", self.name, value))

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be numeric")

        if not 0 <= value <= 100:
            raise ValueError(f"{self.name} must be between 0 and 100")

        instance.__dict__[self.storage_name] = float(value)

Now the host class.

Its `__setattr__` has a different responsibility.

It does not validate the percentage itself.

It only records that a write was attempted, then delegates.

In [8]:
class Download:
    progress = Percentage()

    def __init__(self):
        object.__setattr__(self, "trace", [])
        self.progress = 0

    def __setattr__(self, name, value):
        if name != "trace":
            self.trace.append(("host __setattr__", name, value))
        super().__setattr__(name, value)

Let's perform one more assignment and inspect the trace.

In [9]:
d = Download()
d.progress = 42

d.trace

[('host __setattr__', 'progress', 0),
 ('descriptor __set__', 'progress', 0),
 ('host __setattr__', 'progress', 42),
 ('descriptor __set__', 'progress', 42)]

Each assignment to `progress` should generate two trace entries:

- first from the host class;
- then from the descriptor.

Let's verify both the order and the actual storage.

In [10]:
assert d.trace == [
    ("host __setattr__", "progress", 0),
    ("descriptor __set__", "progress", 0),
    ("host __setattr__", "progress", 42),
    ("descriptor __set__", "progress", 42),
]

assert d.progress == 42.0
assert d.__dict__["_progress"] == 42.0
assert "progress" not in d.__dict__

d.__dict__

{'trace': [('host __setattr__', 'progress', 0),
  ('descriptor __set__', 'progress', 0),
  ('host __setattr__', 'progress', 42),
  ('descriptor __set__', 'progress', 42)],
 '_progress': 42.0}

That last point matters.

The public name `progress` is managed by the descriptor.

The descriptor chose to store the actual value under `_progress`.

So the visible attribute name and the physical storage name do not have to be the same.

## Problem 4 — Failed descriptor writes should not corrupt state

Let's extend the previous idea.

Suppose a managed field already contains a valid value.

Then someone tries to assign an invalid value.

What should happen to the old value?

A well-behaved setter should validate **before** replacing the stored value.

We already wrote the descriptor that way.

So we can test the behavior directly.

In [11]:
before = d.progress

try:
    d.progress = 150
except ValueError as ex:
    print(ex)

after = d.progress

(before, after)

progress must be between 0 and 100


(42.0, 42.0)

The failed assignment entered `__setattr__` and also entered the descriptor's `__set__`.

But because validation failed before storage, the previous value remained intact.

In [12]:
assert before == 42.0
assert after == 42.0

d.trace[-2:]

[('host __setattr__', 'progress', 150),
 ('descriptor __set__', 'progress', 150)]

This suggests another useful separation of responsibilities:

- the outer `__setattr__` can handle cross-cutting behavior such as tracing;
- the descriptor can own field-specific validation;
- failed validation can stop the write before storage changes.

## Problem 5 — Bootstrap a hook that depends on internal state

There is a subtle initialization problem that appears often in real classes.

Suppose `__setattr__` wants to check an internal flag named `_locked`.

But during `__init__`, `_locked` does not exist yet.

If `__setattr__` blindly tries to read it, initialization can fail before the object is usable.

We will solve this with an explicit bootstrap phase.

The class will begin unlocked.

After initialization, public attributes become immutable.

Private attributes remain available for internal maintenance.

In [13]:
class LockedProfile:
    def __init__(self, username, level):
        object.__setattr__(self, "_locked", False)

        self.username = username
        self.level = level

        object.__setattr__(self, "_locked", True)

    def __setattr__(self, name, value):
        locked = object.__getattribute__(self, "_locked")

        if locked and not name.startswith("_"):
            raise AttributeError("public attributes are locked")

        super().__setattr__(name, value)

The important step is the very first line of `__init__`.

We install `_locked` with `object.__setattr__` before any normal public assignment is attempted.

Now the custom hook can safely read the flag.

In [14]:
profile = LockedProfile("ada", 5)

profile.__dict__

{'_locked': True, 'username': 'ada', 'level': 5}

After initialization, the public fields are locked.

In [15]:
try:
    profile.level = 6
except AttributeError as ex:
    print(ex)

assert profile.level == 5

public attributes are locked


But internal implementation state can still be changed deliberately.

In [16]:
profile._last_checked = "today"

assert profile._last_checked == "today"
profile.__dict__

{'_locked': True, 'username': 'ada', 'level': 5, '_last_checked': 'today'}

## Problem 6 — Invalidate a cache when source attributes change

Now let's use `__setattr__` for something more dynamic.

Suppose an object caches an expensive derived value.

Whenever one of the source attributes changes, the cache must be invalidated.

This is a good use for a write interceptor because the invalidation rule is about **assignments**, not just about reads.

We will model a rectangle.

The `area` property caches its result.

Any assignment to `width` or `height` must clear the cache.

In [17]:
class CachedRectangle:
    def __init__(self, width, height):
        object.__setattr__(self, "_area_cache", None)
        self.width = width
        self.height = height

    def __setattr__(self, name, value):
        if name in {"width", "height"}:
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError(f"{name} must be numeric")
            if value <= 0:
                raise ValueError(f"{name} must be positive")

            object.__setattr__(self, "_area_cache", None)

        super().__setattr__(name, value)

    @property
    def area(self):
        cached = object.__getattribute__(self, "_area_cache")

        if cached is None:
            cached = self.width * self.height
            object.__setattr__(self, "_area_cache", cached)

        return cached

Let's calculate the area once.

That should populate `_area_cache`.

In [18]:
rect = CachedRectangle(4, 5)

first_area = rect.area

(first_area, rect._area_cache)

(20, 20)

Now change `width`.

The assignment goes through `__setattr__`, which clears the cache before storing the new width.

In [19]:
rect.width = 10

rect._area_cache

The cache is empty again.

The next read recomputes it from the new dimensions.

In [20]:
second_area = rect.area

assert first_area == 20
assert second_area == 50
assert rect._area_cache == 50

(second_area, rect._area_cache)

(50, 50)

This is a useful advanced pattern:

> Write hooks can maintain invariants between primary state and derived state.

The important part is that internal cache writes bypass the public policy intentionally.

## Problem 7 — Cooperative inheritance and `super().__setattr__`

The source notebook delegates to `super().__setattr__`.

That becomes especially important with multiple inheritance.

If several classes want to participate in the write path, each one should do one job and then call `super()` exactly once.

Let's build two mixins.

The first records every public write.

The second normalizes string values by stripping whitespace.

Neither mixin knows or cares which class comes next in the method resolution order.

In [21]:
class WriteTraceMixin:
    def __setattr__(self, name, value):
        if not name.startswith("_"):
            self._events.append(("trace", name, value))
        super().__setattr__(name, value)


class StripStringsMixin:
    def __setattr__(self, name, value):
        if isinstance(value, str):
            value = value.strip()
        super().__setattr__(name, value)

Now combine the two mixins.

We have to bootstrap `_events` first because the tracing mixin depends on it.

In [22]:
class Contact(WriteTraceMixin, StripStringsMixin):
    def __init__(self, name, city):
        object.__setattr__(self, "_events", [])
        self.name = name
        self.city = city

Let's see the result.

In [23]:
contact = Contact("  Ada Lovelace  ", "  London  ")

contact.__dict__

{'_events': [('trace', 'name', '  Ada Lovelace  '),
  ('trace', 'city', '  London  ')],
 'name': 'Ada Lovelace',
 'city': 'London'}

The trace records the original values because `WriteTraceMixin` appears before `StripStringsMixin` in the MRO.

The stored values are trimmed because the write continues through the next class.

In [24]:
assert contact.name == "Ada Lovelace"
assert contact.city == "London"

assert contact._events == [
    ("trace", "name", "  Ada Lovelace  "),
    ("trace", "city", "  London  "),
]

Contact.__mro__

(__main__.Contact,
 __main__.WriteTraceMixin,
 __main__.StripStringsMixin,
 object)

Now reverse the mixin order and observe the difference.

This is not an error.

It shows that cooperative hooks form a chain, and the MRO determines the order in which each policy sees the value.

In [25]:
class ContactNormalizedFirst(StripStringsMixin, WriteTraceMixin):
    def __init__(self, name):
        object.__setattr__(self, "_events", [])
        self.name = name


contact2 = ContactNormalizedFirst("  Grace Hopper  ")

contact2._events

[('trace', 'name', 'Grace Hopper')]

This time the tracing mixin receives the already-normalized string.

In [26]:
assert contact2.name == "Grace Hopper"
assert contact2._events == [
    ("trace", "name", "Grace Hopper"),
]

## Problem 8 — Notify observers only after a successful write

Let's add side effects.

Suppose other parts of a program can subscribe to changes.

A write should:

1. validate;
2. store the new value;
3. notify listeners **after** the storage succeeds.

If validation fails, no notification should be sent.

We will use a small observable settings object.

The callback receives the attribute name, old value, and new value.

In [27]:
_NOT_SET = object()

class ObservableSettings:
    def __init__(self):
        object.__setattr__(self, "_listeners", [])

    def subscribe(self, callback):
        self._listeners.append(callback)

    def __setattr__(self, name, value):
        if name.startswith("_"):
            return super().__setattr__(name, value)

        if name == "volume":
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError("volume must be an integer")
            if not 0 <= value <= 10:
                raise ValueError("volume must be between 0 and 10")

        old = self.__dict__.get(name, _NOT_SET)

        super().__setattr__(name, value)

        for listener in tuple(self._listeners):
            listener(name, old, value)

Now let's register a listener.

We use a plain list as our event collector.

In [28]:
events = []

def record_change(name, old, new):
    events.append((name, old, new))


settings = ObservableSettings()
settings.subscribe(record_change)

settings.volume = 4
settings.theme = "dark"

events

[('volume', <object at 0x23536965980>, 4),
 ('theme', <object at 0x23536965980>, 'dark')]

A failed write should not appear in the event list because validation happens before storage and before notification.

In [29]:
try:
    settings.volume = 99
except ValueError as ex:
    print(ex)

assert settings.volume == 4
assert len(events) == 2

events

volume must be between 0 and 10


[('volume', <object at 0x23536965980>, 4),
 ('theme', <object at 0x23536965980>, 'dark')]

This gives us another ordering rule:

> If observers should see only successful state changes, notify them **after** the delegated write succeeds.

## Problem 9 — Route one public assignment into several real fields

Sometimes a convenient public attribute is really an input interface.

For example, assigning:

```python
person.full_name = "Ada Lovelace"
```

might update `first_name` and `last_name`.

The tricky part is that writing those two attributes from inside `__setattr__` can re-enter the hook.

That is not automatically wrong, but we need to decide deliberately whether the nested assignments should go through the public policy again.

For this exercise, we **do** want `first_name` and `last_name` to go through the normal policy.

So we will route `full_name` into two ordinary assignments and return immediately.

All other names will follow the normal path.

In [30]:
class RoutedName:
    def __setattr__(self, name, value):
        if name == "full_name":
            if not isinstance(value, str):
                raise TypeError("full_name must be text")

            parts = value.strip().split(maxsplit=1)

            if len(parts) != 2:
                raise ValueError("full_name must contain first and last name")

            self.first_name = parts[0]
            self.last_name = parts[1]
            return

        if name in {"first_name", "last_name"}:
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"{name} must be non-empty text")
            value = value.strip().title()

        super().__setattr__(name, value)

Let's use the routing attribute.

In [31]:
person = RoutedName()
person.full_name = "  ada lovelace  "

person.__dict__

{'first_name': 'Ada', 'last_name': 'Lovelace'}

Notice that `full_name` itself was never stored.

It acted as a write interface that produced two real attributes.

In [32]:
assert person.first_name == "Ada"
assert person.last_name == "Lovelace"
assert "full_name" not in person.__dict__

This is controlled re-entry.

`self.first_name = ...` and `self.last_name = ...` call `__setattr__` again, but they take a different branch and eventually delegate normally.

So recursion is only a bug when the write path loops without making progress toward a terminating branch.

## Problem 10 — Class writes need the metaclass, not the instance hook

The source notebook makes an important distinction:

- `obj.x = value` is an instance write;
- `SomeClass.x = value` is a write to the **class object**.

An instance-level `__setattr__` does not control the second form.

For that, we need a metaclass.

Let's create a metaclass that records changes to public class configuration.

We will ignore private names so Python and our own bookkeeping can work normally.

In [33]:
class ConfigMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        type.__setattr__(cls, "_class_write_log", [])
        return cls

    def __setattr__(cls, name, value):
        if not name.startswith("_"):
            cls._class_write_log.append((name, value))
        super().__setattr__(name, value)

Now define a class using that metaclass.

The instance also has its own `__setattr__`, so we can see that the two write channels are independent.

In [34]:
class Service(metaclass=ConfigMeta):
    timeout = 30

    def __init__(self):
        object.__setattr__(self, "instance_log", [])

    def __setattr__(self, name, value):
        if name != "instance_log":
            self.instance_log.append((name, value))
        super().__setattr__(name, value)

First perform a class write.

In [35]:
Service.timeout = 45
Service.region = "eu"

Service._class_write_log

[('timeout', 45), ('region', 'eu')]

Now perform instance writes with the same names.

In [36]:
service = Service()

service.timeout = 5
service.region = "local"

service.instance_log

[('timeout', 5), ('region', 'local')]

The two logs should contain different events.

This shows that the receiver of the assignment matters:

- the class object's type handles class writes;
- the instance's class handles instance writes.

In [37]:
assert Service._class_write_log == [
    ("timeout", 45),
    ("region", "eu"),
]

assert service.instance_log == [
    ("timeout", 5),
    ("region", "local"),
]

assert Service.timeout == 45
assert service.timeout == 5

## Problem 11 — Enforce a class-level naming rule with a metaclass

Now let's turn class-write interception into a real policy.

Suppose all attributes whose names start with `LIMIT_` must always be non-negative integers.

The rule applies when modifying the class at runtime.

We will extend the metaclass.

The validation happens before delegation, so invalid writes never reach the class dictionary.

In [38]:
class LimitMeta(type):
    def __setattr__(cls, name, value):
        if name.startswith("LIMIT_"):
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"{name} must be an integer")
            if value < 0:
                raise ValueError(f"{name} cannot be negative")

        super().__setattr__(name, value)


class ApiPolicy(metaclass=LimitMeta):
    LIMIT_RETRIES = 3
    LIMIT_CONNECTIONS = 100

Valid runtime class writes work normally.

In [39]:
ApiPolicy.LIMIT_RETRIES = 5
ApiPolicy.description = "production policy"

(ApiPolicy.LIMIT_RETRIES, ApiPolicy.description)

(5, 'production policy')

Now try an invalid class write.

In [40]:
try:
    ApiPolicy.LIMIT_CONNECTIONS = -1
except ValueError as ex:
    print(ex)

assert ApiPolicy.LIMIT_CONNECTIONS == 100

LIMIT_CONNECTIONS cannot be negative


One subtle point remains.

The assignments inside the **class body** occur while the namespace dictionary is being built, before the final class object exists.

So the runtime metaclass `__setattr__` hook does not validate those original class-body values.

If we wanted to validate the initial class namespace too, we would add logic to the metaclass's `__new__` or `__init__`.

## Problem 12 — Validate both class creation and later class mutation

Let's solve that last problem.

We want the `LIMIT_` rule to apply in two places:

- when the class is first created;
- when those class attributes are later changed.

A clean way to avoid duplicating the validation logic is to move it into a helper on the metaclass.

In [41]:
class StrictLimitMeta(type):
    @staticmethod
    def _validate_limit(name, value):
        if name.startswith("LIMIT_"):
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"{name} must be an integer")
            if value < 0:
                raise ValueError(f"{name} cannot be negative")

    def __new__(mcls, name, bases, namespace):
        for attr_name, value in namespace.items():
            mcls._validate_limit(attr_name, value)

        return super().__new__(mcls, name, bases, namespace)

    def __setattr__(cls, name, value):
        type(cls)._validate_limit(name, value)
        super().__setattr__(name, value)

A valid class can now be created normally.

In [42]:
class WorkerPool(metaclass=StrictLimitMeta):
    LIMIT_WORKERS = 8
    LIMIT_QUEUE = 1000


WorkerPool.LIMIT_WORKERS

8

And later writes are still checked.

In [43]:
WorkerPool.LIMIT_WORKERS = 16

try:
    WorkerPool.LIMIT_QUEUE = -50
except ValueError as ex:
    print(ex)

assert WorkerPool.LIMIT_WORKERS == 16
assert WorkerPool.LIMIT_QUEUE == 1000

LIMIT_QUEUE cannot be negative


We can even demonstrate that invalid initial class definitions are rejected.

We will define the class inside a `try` block so the notebook can continue running.

In [44]:
try:
    class BrokenPool(metaclass=StrictLimitMeta):
        LIMIT_WORKERS = -3
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError LIMIT_WORKERS cannot be negative


## Problem 13 — Use `__slots__` and still customize writes

So far most examples have had an instance `__dict__`.

But `__setattr__` does not require one.

Let's make a slotted class.

This is interesting because direct tricks like:

```python
self.__dict__[name] = value
```

would not work here at all.

The correct approach is still to delegate to the normal attribute machinery.

The slot descriptors will handle the storage.

In [45]:
class Vector2D:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __setattr__(self, name, value):
        if name in {"x", "y"}:
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError(f"{name} must be numeric")

        super().__setattr__(name, value)

Let's create an instance and confirm that it really has no instance dictionary.

In [46]:
vector = Vector2D(1, 2)
vector.x = 10

assert vector.x == 10
assert vector.y == 2
assert not hasattr(vector, "__dict__")

(vector.x, vector.y)

(10, 2)

Trying to add an arbitrary attribute fails because there is no slot for it.

In [47]:
try:
    vector.label = "point"
except AttributeError as ex:
    print(ex)

'Vector2D' object has no attribute 'label'


This is one reason direct `__dict__` manipulation is usually a poor substitute for proper delegation.

Delegating with `super().__setattr__` works with ordinary dictionaries, descriptors, properties, and slots.

## Problem 14 — A small state machine controlled by writes

For the final problem, let's combine several ideas.

Imagine an order with three states:

- `draft`
- `submitted`
- `closed`

While the order is in `draft`, public fields may be edited.

After submission, only `status` may change.

Once closed, no public attribute may be changed.

This is a write policy based on the object's current state.

The main difficulty is initialization.

The `status` field must exist before the hook can use it to decide what is allowed.

In [48]:
class Order:
    VALID_STATES = {"draft", "submitted", "closed"}

    def __init__(self, item, quantity):
        object.__setattr__(self, "status", "draft")
        self.item = item
        self.quantity = quantity

    def __setattr__(self, name, value):
        current_status = object.__getattribute__(self, "status")

        if name == "status":
            if value not in self.VALID_STATES:
                raise ValueError(f"invalid status: {value!r}")

            allowed_transitions = {
                "draft": {"draft", "submitted"},
                "submitted": {"submitted", "closed"},
                "closed": {"closed"},
            }

            if value not in allowed_transitions[current_status]:
                raise ValueError(
                    f"cannot move from {current_status!r} to {value!r}"
                )

            return super().__setattr__(name, value)

        if current_status == "submitted":
            raise AttributeError(
                "submitted orders may only change status"
            )

        if current_status == "closed":
            raise AttributeError("closed orders are immutable")

        if name == "quantity":
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError("quantity must be an integer")
            if value <= 0:
                raise ValueError("quantity must be positive")

        super().__setattr__(name, value)

Let's begin in the draft state.

Normal edits should work.

In [49]:
order = Order("keyboard", 1)

order.quantity = 2
order.item = "mechanical keyboard"

order.__dict__

{'status': 'draft', 'item': 'mechanical keyboard', 'quantity': 2}

Now submit the order.

After that, ordinary public edits should be rejected.

In [50]:
order.status = "submitted"

try:
    order.quantity = 3
except AttributeError as ex:
    print(ex)

assert order.quantity == 2
assert order.status == "submitted"

submitted orders may only change status


But the allowed state transition to `closed` still works.

In [51]:
order.status = "closed"

assert order.status == "closed"

And once closed, even attempting to go back to `submitted` is rejected by the transition rule.

In [52]:
try:
    order.status = "submitted"
except ValueError as ex:
    print(ex)

assert order.status == "closed"

cannot move from 'closed' to 'submitted'


This final example shows how far the basic mechanism can be taken.

The hook is still doing the same fundamental thing as in the source notebook:

1. intercept the assignment;
2. apply a policy;
3. either reject the write or delegate it.

The complexity comes from the policy, not from a different attribute mechanism.

## Summary

Let's collect the main ideas from the problems.

`__setattr__` is the central instance-level hook for attribute assignments demonstrated in the source notebook.

When we override it, every normal instance assignment reaches that hook.

If we want the normal attribute machinery to continue, we delegate.

Most commonly:

```python
super().__setattr__(name, value)
```

That preserves descriptor behavior and cooperative inheritance.

When we intentionally need to bypass our custom hook for internal bookkeeping, we can call:

```python
object.__setattr__(self, name, value)
```

That technique is especially useful while bootstrapping internal state or updating caches and counters.

Data descriptors remain part of the write path.

If delegated assignment targets a data descriptor, the descriptor's `__set__` method handles the managed field.

Properties with setters are one familiar example.

Class writes are a separate channel.

For:

```python
SomeClass.x = value
```

the receiver is the class object itself.

To customize that operation, we define `__setattr__` on the metaclass.

And finally, recursion is not simply “calling `__setattr__` twice”.

The real problem is a write path that keeps routing back to itself without reaching a terminating branch.

Controlled re-entry can be useful, but accidental self-assignment such as:

```python
def __setattr__(self, name, value):
    self.some_name = value
```

usually creates infinite recursion.

## Final challenge

Without looking back at the solutions, try to design a class that combines all of the following:

- a data descriptor for one validated field;
- an outer `__setattr__` audit log;
- an internal cache invalidated by selected writes;
- a metaclass that validates one family of class-level configuration attributes;
- cooperative behavior with a normalization mixin.

The key question for every line should be:

> Is this a public write that should go through the policy, or an internal implementation write that should deliberately bypass it?

That distinction is the foundation of safe attribute-write customization.